# Причины расхождений: Excel vs merchants (февраль)

Тетрадка отвечает на вопрос: почему `only_excel` и `only_datamart` не совпадают.

Основная логика:
1. Сравниваем множества `agr_id`, `inn`, `c_agr_number`.
2. Для `only_excel agr_id` раскладываем причины:
   - нет в `agreements`;
   - `acq_class` не `SA`;
   - договор не активен на 1-е число месяца;
   - формально должен быть в периметре, но отсутствует в `stg__..._merchants`.
3. Для `only_datamart agr_id` даем техническую сегментацию относительно Excel.

Подключение — в формате, который вы используете (без `LAKE_USER/LAKE_PASSWORD`).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026.xlsx"
excel_agr_col = "ID договора"
excel_inn_col = "ИНН"
excel_contract_col = "Номер договора"

month_start = "2026-02-01"  # правило активного клиента на 1-е число
merchants_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    return s if re.fullmatch(r"\d+", s) else None

def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s

def to_sql_in_list(values):
    return ','.join([f"'{v}'" for v in values])

## 1) Excel и merchants-срез на 1-е число месяца

In [ ]:
df_excel = pd.read_excel(excel_path)

excel_norm = pd.DataFrame({
    'agr_id': df_excel[excel_agr_col].apply(normalize_id),
    'inn': df_excel[excel_inn_col].apply(normalize_id),
    'c_agr_number': df_excel[excel_contract_col].apply(normalize_str),
}).dropna(how='all')

with imp:
    dm = imp.fetch(f"""
        select
            cast(agr_id as string) as agr_id,
            cast(inn as string) as inn,
            cast(c_agr_number as string) as c_agr_number,
            cast(d_valid_from as date) as d_valid_from,
            cast(d_valid_to as date) as d_valid_to
        from {merchants_table}
        where cast(d_valid_from as date) <= cast('{month_start}' as date)
          and (
                d_valid_to is null
                or cast(d_valid_to as date) >= cast('{month_start}' as date)
              )
    """)

dm_norm = pd.DataFrame({
    'agr_id': dm['agr_id'].apply(normalize_id),
    'inn': dm['inn'].apply(normalize_id),
    'c_agr_number': dm['c_agr_number'].apply(normalize_str),
}).dropna(how='all')

print('excel_rows =', len(excel_norm))
print('merchants_rows_active_on_1st =', len(dm_norm))

In [ ]:
set_excel_agr = set(excel_norm['agr_id'].dropna().tolist())
set_dm_agr = set(dm_norm['agr_id'].dropna().tolist())

set_excel_inn = set(excel_norm['inn'].dropna().tolist())
set_dm_inn = set(dm_norm['inn'].dropna().tolist())

set_excel_num = set(excel_norm['c_agr_number'].dropna().tolist())
set_dm_num = set(dm_norm['c_agr_number'].dropna().tolist())

summary = pd.DataFrame([
    {'entity': 'agr_id', 'excel_unique': len(set_excel_agr), 'merchants_unique': len(set_dm_agr), 'intersect': len(set_excel_agr & set_dm_agr), 'only_excel': len(set_excel_agr - set_dm_agr), 'only_merchants': len(set_dm_agr - set_excel_agr)},
    {'entity': 'inn', 'excel_unique': len(set_excel_inn), 'merchants_unique': len(set_dm_inn), 'intersect': len(set_excel_inn & set_dm_inn), 'only_excel': len(set_excel_inn - set_dm_inn), 'only_merchants': len(set_dm_inn - set_excel_inn)},
    {'entity': 'c_agr_number', 'excel_unique': len(set_excel_num), 'merchants_unique': len(set_dm_num), 'intersect': len(set_excel_num & set_dm_num), 'only_excel': len(set_excel_num - set_dm_num), 'only_merchants': len(set_dm_num - set_excel_num)},
])
display(summary)

only_excel_agr = sorted(list(set_excel_agr - set_dm_agr))
only_dm_agr = sorted(list(set_dm_agr - set_excel_agr))
print('only_excel_agr_cnt =', len(only_excel_agr))
print('only_dm_agr_cnt =', len(only_dm_agr))

## 2) Причины для `only_excel agr_id`

In [ ]:
def fetch_agreements_for_ids(agr_ids, chunk_size=500):
    """Тянем минимальные признаки из agreements по chunk IN-list."""
    all_parts = []
    for i in range(0, len(agr_ids), chunk_size):
        chunk = agr_ids[i:i+chunk_size]
        if not chunk:
            continue
        in_list = to_sql_in_list(chunk)
        sql = f"""
            select
                cast(a.abs_agr_id as string) as agr_id,
                a.acq_class,
                cast(a.d_valid_from as date) as d_valid_from,
                cast(a.d_valid_to as date) as d_valid_to,
                cast(c.c_inn as string) as inn,
                cast(a.c_agr_number as string) as c_agr_number
            from ods_alpha.scd1_agreements a
            left join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where cast(a.abs_agr_id as string) in ({in_list})
        """
        with imp:
            part = imp.fetch(sql)
        all_parts.append(part)
    if not all_parts:
        return pd.DataFrame(columns=['agr_id', 'acq_class', 'd_valid_from', 'd_valid_to', 'inn', 'c_agr_number'])
    return pd.concat(all_parts, ignore_index=True)

agr_check = pd.DataFrame({'agr_id': only_excel_agr})

if len(only_excel_agr) > 0:
    agr_info = fetch_agreements_for_ids(only_excel_agr)
    if len(agr_info) > 0:
        # Если дублей несколько, берем первую запись (для диагностики причин достаточно)
        agr_info = agr_info.sort_values(['agr_id']).drop_duplicates(subset=['agr_id'], keep='first')
    agr_check = agr_check.merge(agr_info, on='agr_id', how='left')
else:
    agr_check['acq_class'] = None
    agr_check['d_valid_from'] = None
    agr_check['d_valid_to'] = None
    agr_check['inn'] = None
    agr_check['c_agr_number'] = None

agr_check['in_agreements'] = agr_check['acq_class'].notna()
agr_check['is_sa'] = agr_check['acq_class'].astype(str).eq('SA')
agr_check['active_on_month_start'] = (
    pd.to_datetime(agr_check['d_valid_from'], errors='coerce').dt.date <= pd.to_datetime(month_start).date()
) & (
    agr_check['d_valid_to'].isna() |
    (pd.to_datetime(agr_check['d_valid_to'], errors='coerce').dt.date >= pd.to_datetime(month_start).date())
)

def detect_reason(row):
    if not row['in_agreements']:
        return '01_not_found_in_agreements'
    if not row['is_sa']:
        return '02_acq_class_not_SA'
    if not row['active_on_month_start']:
        return '03_inactive_on_month_start'
    return '04_should_be_in_scope_but_missing_in_merchants_snapshot'

agr_check['reason'] = agr_check.apply(detect_reason, axis=1)

reason_summary = (
    agr_check.groupby('reason', as_index=False)
    .agg(cnt_agr=('agr_id', 'count'))
    .sort_values('cnt_agr', ascending=False)
)

display(reason_summary)
display(agr_check.head(100))

## 3) Тех.сегментация для `only_merchants agr_id`

In [ ]:
only_dm_df = pd.DataFrame({'agr_id': only_dm_agr})

dm_sample = (
    dm_norm[['agr_id', 'inn', 'c_agr_number']]
    .dropna(subset=['agr_id'])
    .drop_duplicates(subset=['agr_id'], keep='first')
)
only_dm_df = only_dm_df.merge(dm_sample, on='agr_id', how='left')

only_dm_df['inn_in_excel'] = only_dm_df['inn'].isin(set_excel_inn)
only_dm_df['agr_num_in_excel'] = only_dm_df['c_agr_number'].isin(set_excel_num)

def dm_reason(row):
    if row['inn_in_excel'] and row['agr_num_in_excel']:
        return 'A_inn_and_contract_exist_in_excel_but_agr_id_diff'
    if row['inn_in_excel'] and not row['agr_num_in_excel']:
        return 'B_inn_exists_but_contract_number_missing_in_excel'
    if (not row['inn_in_excel']) and row['agr_num_in_excel']:
        return 'C_contract_number_exists_but_inn_missing_in_excel'
    return 'D_inn_and_contract_not_found_in_excel'

only_dm_df['reason'] = only_dm_df.apply(dm_reason, axis=1)

only_dm_summary = (
    only_dm_df.groupby('reason', as_index=False)
    .agg(cnt_agr=('agr_id', 'count'))
    .sort_values('cnt_agr', ascending=False)
)

display(only_dm_summary)
display(only_dm_df.head(100))

## 4) Экспорт топ-расхождений (опционально)

Если нужно отправить коллегам, раскомментируйте строки сохранения.

In [ ]:
display(reason_summary)
display(only_dm_summary)

# agr_check.to_excel('/home/jovyan/documents/Equaring/Проверки/only_excel_agr_reasons.xlsx', index=False)
# only_dm_df.to_excel('/home/jovyan/documents/Equaring/Проверки/only_merchants_agr_reasons.xlsx', index=False)
# reason_summary.to_excel('/home/jovyan/documents/Equaring/Проверки/reason_summary_only_excel.xlsx', index=False)
# only_dm_summary.to_excel('/home/jovyan/documents/Equaring/Проверки/reason_summary_only_merchants.xlsx', index=False)

print('Done')